# Occupancy Deployment Model V1

This notebook is the clear human-facing workflow for the active occupancy model.

The model we deploy is the XGBoost + HistGradientBoosting blend with price-positioning features. Those features compare the listing's actual price against a leakage-safe fair price prediction from the price model.

## Cell 1 - Locate Project Files

This cell finds the deployment folder, the model artifact folder, and the occupancy training script.

In [1]:
from pathlib import Path
import json
import runpy

def find_deployment_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "code files" / "ml" / "deployment_models",
        Path.cwd().parent / "deployment_models",
    ]
    for candidate in candidates:
        if (candidate / "occupancy_deployment_model_v1.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find occupancy_deployment_model_v1.py")

DEPLOYMENT_DIR = find_deployment_dir()
PROJECT_ROOT = DEPLOYMENT_DIR.parents[2]
ARTIFACT_DIR = PROJECT_ROOT / "data" / "gold" / "modeling" / "deployment_models"
OCCUPANCY_SCRIPT = DEPLOYMENT_DIR / "occupancy_deployment_model_v1.py"

print("Deployment folder:", DEPLOYMENT_DIR)
print("Artifact folder:", ARTIFACT_DIR)
print("Occupancy script:", OCCUPANCY_SCRIPT)

Deployment folder: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models
Artifact folder: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\data\gold\modeling\deployment_models
Occupancy script: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\occupancy_deployment_model_v1.py


## Cell 2 - Rebuild The Occupancy Deployment Model

Run this cell when you want to retrain and resave the active occupancy deployment artifact. The script asks for the MySQL password using `getpass`, so the password stays hidden.

In [5]:
runpy.run_path(str(OCCUPANCY_SCRIPT), run_name="__main__")

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: NO)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## Cell 3 - Read The Final Scores

This cell reads the deployment metadata and displays the final test metrics for every tested occupancy variant.

In [4]:
import pandas as pd

metadata_path = ARTIFACT_DIR / "occupancy_deployment_model_v1_metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

occupancy_metrics = pd.DataFrame(metadata["test_reference_metrics"])
occupancy_metrics.sort_values(["r2", "rmse", "mae"], ascending=[False, True, True])

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\mario\\Documents\\DSMLAISL LEARNING PATH\\Portugal-Rental-Investment-\\data\\gold\\modeling\\deployment_models\\occupancy_deployment_model_v1_metadata.json'

## Cell 4 - Inspect Price-Gap Feature Importance

This cell shows whether the fair-price gap features helped. `price_gap_pct` is the key feature because it tells the model whether the listing is cheap, fair, or expensive relative to similar listings.

In [ ]:
importance = pd.DataFrame(metadata["price_gap_feature_importance"])
importance.sort_values("mean_r2_drop_when_shuffled", ascending=False)

## Cell 5 - Confirm The Saved Artifact

This cell loads the `.joblib` bundle and confirms the selected model. This is the object that later deployment code should load for occupancy prediction.

In [ ]:
import joblib

model_path = ARTIFACT_DIR / "occupancy_deployment_model_v1.joblib"
occupancy_bundle = joblib.load(model_path)

print("Artifact:", model_path)
print("Bundle type:", occupancy_bundle.get("bundle_type"))
print("Selected model:", occupancy_bundle.get("selected_model"))
print("Depends on price model:", occupancy_bundle.get("fair_price_model_stem"))
print("Feature count:", len(occupancy_bundle.get("feature_columns", [])))

## Cell 6 - Deployment Notes

The active occupancy model is selected because it improves R2 without worsening RMSE or MAE. The CSV files from testing are not required for deployment because the report, registry, metadata, and artifact contain the important information.